In [ ]:
import pandas as pd
import numpy as np
import pickle
import re
import sys
from pathlib import Path

# ============================================================================
# SUBMISSÃO A (ALTA CONFIANÇA)
# Idealmente: Modelo PyTorch GRU (F1≈0.83, Acc≈0.83)
# Fallback: Modelo NumPy DNN (F1≈0.58, Acc≈0.59) se PyTorch não disponível
# ============================================================================

PROJECT_ROOT = Path("/Users/afonso/llm-detector")
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Pasta 'src' não encontrada em {PROJECT_ROOT}")
sys.path.insert(0, str(PROJECT_ROOT))

from src.neuralnet import NeuralNetwork

class NeuralNetworkTracked(NeuralNetwork):
    pass
sys.modules["__main__"].NeuralNetworkTracked = NeuralNetworkTracked

PATH_DATASET_TESTE = PROJECT_ROOT / "data" / "subm2.csv"
df_teste = pd.read_csv(PATH_DATASET_TESTE, sep=None, engine="python")

# Normalizar colunas para evitar problemas de BOM/capitalização
df_teste.columns = [str(c).replace("\ufeff", "").strip() for c in df_teste.columns]
cols_lower = {c.lower(): c for c in df_teste.columns}
if "id" in cols_lower and "ID" not in df_teste.columns:
    df_teste = df_teste.rename(columns={cols_lower["id"]: "ID"})
if "text" in cols_lower and "Text" not in df_teste.columns:
    df_teste = df_teste.rename(columns={cols_lower["text"]: "Text"})
if "Text" not in df_teste.columns:
    raise ValueError("Coluna 'Text' não encontrada no ficheiro de teste.")
if "ID" not in df_teste.columns:
    df_teste.insert(0, "ID", np.arange(1, len(df_teste) + 1, dtype=np.int64))
id_like = [c for c in df_teste.columns if c.lower() == "id"]
if len(id_like) > 1:
    keep = id_like[0]
    df_teste = df_teste.drop(columns=id_like[1:])
    if keep != "ID":
        df_teste = df_teste.rename(columns={keep: "ID"})

# Carregar modelo e artefatos
with open(PROJECT_ROOT / "modelo_numpy_artefactos.pkl", "rb") as f:
    artefactos = pickle.load(f)
    le = artefactos["label_encoder"]
    vocab = artefactos["vocab"]
    idf = artefactos["idf"]
    net_np = artefactos["model"]

def clean_text_np(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.split()

def transform_tfidf(tokens, vocab, idf):
    vec = np.zeros(len(vocab))
    for token in tokens:
        if token in vocab:
            vec[vocab[token]] = idf[vocab[token]]
    return vec

# Processar exemplos de teste
texts_clean = [clean_text_np(text) for text in df_teste["Text"].values]
X_teste_tfidf = np.array([transform_tfidf(tokens, vocab, idf) for tokens in texts_clean])

# Inferência com modelo NumPy DNN
preds_np_probs = net_np.predict(X_teste_tfidf)
preds_np_idx = np.argmax(preds_np_probs, axis=1)

# A = submissão com alta confiança
df_teste["Label"] = le.inverse_transform(preds_np_idx)

NOME_FICHEIRO_SAIDA = PROJECT_ROOT / "Subm2/subm2-g9-MEI-A.csv"
df_saida = df_teste[["ID", "Label"]].copy()
df_saida.to_csv(NOME_FICHEIRO_SAIDA, index=False, sep=";")

df_saida.head()

Processando exemplos de teste...
Executando inferência (Modelo NumPy)...


,ID,Label
0,D2-101,Anthropic
1,D2-102,Anthropic
2,D2-103,Anthropic
3,D2-104,Anthropic
4,D2-105,Anthropic
